In [4]:
import os 
import numpy as np
import random
import math
import json
from functools import partial
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data 
import torch.optim as optim

import pytorch_lightning as pl


ModuleNotFoundError: No module named 'numpy'

In [ ]:
def scaled_dot_product(Q,K,V,Mask=None):
    '''
    This is just the frindely attention helper function, so basic 

    atttention = softmax(Q@K^t)/sqrt(d)@V
    '''

    d_k = Q.size(-1) # for the q matrix (Batchsize, Head, Sequence, Head_Dimension)
    attention_logits = Q@K.transpose(-2,-1) # Eg: Q(1, 1, 4, 64) K(1, 1, 64, 4) Basically the torch.matmul leave out the batchsize and Head; just conisder the Sequence and the Head_dimenion
    attention_logits = attention_logits/math.sqrt(d_k) # # Scale attention scores by sqrt(d_k) to prevent large logits and softmax saturation.
    if Mask is not None:
        attention_logits = attention_logits.masked_fill(Mask == 0,-9e15) # Causal Attention;
    attention_weight = F.softmax(attention_logits,dim=-1) # attention_logits:(1 ,1, 4, 4)
    contex_vector = attention_weight@V

    return contex_vector,attention_weight


In [ ]:
seq_len, d_k = 3, 2
pl.seed_everything(42)
q = torch.randn(seq_len, d_k)
k = torch.randn(seq_len, d_k)
v = torch.randn(seq_len, d_k)
values, attention = scaled_dot_product(q, k, v)
print("Q\n", q)
print("K\n", k)
print("V\n", v)
print("Values\n", values)
print("Attention\n", attention)

Seed set to 42


Q
 tensor([[ 0.3367,  0.1288],
        [ 0.2345,  0.2303],
        [-1.1229, -0.1863]])
K
 tensor([[ 2.2082, -0.6380],
        [ 0.4617,  0.2674],
        [ 0.5349,  0.8094]])
V
 tensor([[ 1.1103, -1.6898],
        [-0.9890,  0.9580],
        [ 1.3221,  0.8172]])
Values
 tensor([[ 0.5698, -0.1520],
        [ 0.5379, -0.0265],
        [ 0.2246,  0.5556]])
Attention
 tensor([[0.4028, 0.2886, 0.3086],
        [0.3538, 0.3069, 0.3393],
        [0.1303, 0.4630, 0.4067]])


In [ ]:
## Helper function to support different mask shape 
## We need our output should be in this formate (batch_size, num_head, seq_len, seq_len)
## if 2D: which is (seq_len, seq_len) -> (1,1,seq_len,seq_len)
## if 3D: which (batch_size, seq_len, seq_len) ->. (batch_size, 1 ,seq_len, seq_len)

def expand_mask(mask):
    assert mask.ndim >= 2, "Mask should be atleast 2D (seq_len, seq_len)"
    if mask.ndim ==3:
        mask.unsqueeze(1) # (batch_size, seq_len, seq_len) ->. (batch_size, 1 ,seq_len, seq_len)
    while mask.ndim<4:
        mask.unsqueeze(0) # (seq_len, seq_len) -> (1, seq_len, seq_len) -> (1, 1, seq_len, seq_len)
    return mask

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, input_dim, embed_dim, num_head):
        super().__init__()
        """ 
        Multi Head Attention Block 
        split the Q, K, V matrix into blocks of smaller dim  and process it through scaled dot product,
        and then concatinate at the last

        Input: 
        input_dim - Dimension of the input,
        num_head - number of the heads in the attention block,
        embed_dim - Dimension of the hidden layer, most of the time it is equal to the input dim

        Output: 
        Values - 
        Attention_score_matrix 
        """

        self.embed_dim = embed_dim
        self.num_head = num_head

        assert self.embed_dim % num_head == 0, "Embedding dim modulo with the head dimension must be zero"

        self.head_dim =  self.embed_dim // self.num_head

        self.qkv_proj = nn.Linear(input_dim,3*embed_dim) # Rather than doing the matmul for three different matrix, it is efficient for cuda and other kernal to do in single vector 
        self.output_proj = nn.Linear(embed_dim,input_dim)

        self._reset_parameters()

    def _reset_parameters(self):
        """ 
        Original Transformer initialization, see PyTorch documentation
        What does this function does is rather than starting all the weights form zero it initialize 
        with small random number, other wise all the weights converge to the same area 
        """
        nn.init.xavier_uniform_(self.qkv_proj.weight)
        self.qkv_proj.bias.data.fill_(0)
        nn.init.xavier_uniform_(self.output_proj.weight)
        self.output_proj.bias.data.fill_(0)

    def forward(self, x, mask=None, return_attention = False):
        if mask is not None:
            mask = expand_mask(mask)

        batch_size,seq_len, _ = x.size()
        qkv = self.qkv_proj(x)

        qkv = qkv.reshape(batch_size, seq_len, self.num_head, 3*self.head_dim) #reshaping into seperate q k v 
        qkv = qkv.permute(0,2,1,3)#[Batch_size, Head, Seq_len, head_Dims ] # cuz for the torch matmul it takes the last two variables 
        q, k, v = qkv.chunk(3,dim=-1) 

        values, attention = scaled_dot_product(q,k,v, Mask=mask) 
        values = values.permute(0,2,1,3) # Back to the normal Size [Batch_size, seq_len, num_head, head_dim]
        values = values.reshape(batch_size, seq_len, self.embed_dim)

        output = self.output_proj(values)

        if return_attention:
            return output, attention
        else:
            return output

        



        

: 